# 03 - Inference with the packaged model
Loads `artifacts/model/model.joblib` (preprocessing + classifier in one pipeline) and verifies that predictions are reproducible after reloading.

In [1]:
# Make the repository's src/ importable whether or not `pip install -e .` was run
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)
import platform, socket, datetime
print("Run on host:", socket.gethostname(), "| Python", platform.python_version(),
      "| started", datetime.datetime.now().isoformat(timespec="seconds"))

Run on host: ip-10-3-24-59.ap-south-1.compute.internal | Python 3.12.9 | started 2026-10-16T20:58:48


In [2]:
from heartrisk.model_io import load_model
import json
model = load_model()
print(json.dumps({k: model.metadata[k] for k in ['model_family','model_version','best_params','test_metrics']}, indent=2))

{
  "model_family": "logistic_regression",
  "model_version": "1.0.0+b00a9c9",
  "best_params": {
    "clf__C": 1.0,
    "clf__class_weight": "balanced",
    "clf__penalty": "l2",
    "features__add_hr_ratio": false
  },
  "test_metrics": {
    "test_accuracy": 0.8852459016393442,
    "test_precision": 0.8181818181818182,
    "test_recall": 0.9642857142857143,
    "test_f1": 0.8852459016393442,
    "test_roc_auc": 0.958874458874459,
    "test_specificity": 0.8181818181818182,
    "test_tn": 27,
    "test_fp": 6,
    "test_fn": 1,
    "test_tp": 27
  }
}


In [3]:
sample = json.load(open('samples/sample_request.json'))
model.predict_records([sample])

[{'prediction': 1,
  'label': 'heart disease present',
  'probability_disease': 0.9615,
  'confidence': 0.9615}]

Missing `ca` / `thal` are allowed (imputed exactly as during training):

In [4]:
model.predict_records([{k: v for k, v in sample.items() if k not in ('ca', 'thal')}])

[{'prediction': 0,
  'label': 'no heart disease',
  'probability_disease': 0.3164,
  'confidence': 0.6836}]

### Round-trip check on the cleaned dataset

In [5]:
import joblib, numpy as np
from heartrisk.data import load_clean, split_xy
X, y = split_xy(load_clean())
a = joblib.load('artifacts/model/model.joblib').predict_proba(X)[:, 1]
b = np.array([r['probability_disease'] for r in model.predict_records(X.to_dict('records'))])
print('max |difference| =', float(np.abs(a - b).max()))
assert np.allclose(a, b, atol=1e-4)

max |difference| = 4.95970100654608e-05
